## 1. Import Required Libraries

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## 2. Load and Explore the Data

In [ ]:
# Load dataset
data = load_breast_cancer(as_frame=True)
df = data.frame

print(f"Dataset shape: {df.shape}")
print(f"\nFeatures: {data.feature_names}")
print(f"\nTarget distribution:\n{df['target'].value_counts()}")
df.head()

## 3. Data Preparation

In [ ]:
# Split features and target
X, y = load_breast_cancer(return_X_y=True)

# Split into train and test sets (using fixed random_state for reproducibility)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Scale the features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Training set size: {X_train_scaled.shape[0]}")
print(f"Test set size: {X_test_scaled.shape[0]}")

## 4. Baseline Model - KNN (Original)

In [ ]:
# Baseline KNN with default parameters
knn_baseline = KNeighborsClassifier()
knn_baseline.fit(X_train_scaled, y_train)
baseline_score = knn_baseline.score(X_test_scaled, y_test)

print(f"Baseline KNN Accuracy: {baseline_score:.4f}")

## 5. Model Optimization - Hyperparameter Tuning for KNN

In [ ]:
# Grid search for best KNN parameters
param_grid_knn = {
    'n_neighbors': [3, 5, 7, 9, 11, 15],
    'weights': ['uniform', 'distance'],
    'metric': ['euclidean', 'manhattan', 'minkowski']
}

knn_grid = GridSearchCV(KNeighborsClassifier(), param_grid_knn, cv=5, scoring='accuracy', n_jobs=-1)
knn_grid.fit(X_train_scaled, y_train)

print(f"Best KNN parameters: {knn_grid.best_params_}")
print(f"Best cross-validation score: {knn_grid.best_score_:.4f}")
print(f"Test set accuracy: {knn_grid.score(X_test_scaled, y_test):.4f}")

## 6. Compare Multiple Algorithms

In [ ]:
# Dictionary of models to compare
models = {
    'KNN (Optimized)': knn_grid.best_estimator_,
    'Logistic Regression': LogisticRegression(max_iter=10000, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'SVM': SVC(kernel='rbf', random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, random_state=42)
}

results = {}

for name, model in models.items():
    if name != 'KNN (Optimized)':  # Already fitted
        model.fit(X_train_scaled, y_train)
    
    # Get predictions
    y_pred = model.predict(X_test_scaled)
    
    # Calculate metrics
    accuracy = accuracy_score(y_test, y_pred)
    cv_scores = cross_val_score(model, X_train_scaled, y_train, cv=5)
    
    results[name] = {
        'accuracy': accuracy,
        'cv_mean': cv_scores.mean(),
        'cv_std': cv_scores.std()
    }
    
    print(f"\n{name}:")
    print(f"  Test Accuracy: {accuracy:.4f}")
    print(f"  CV Accuracy: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")

## 7. Visualize Model Comparison

In [ ]:
# Create comparison plot
results_df = pd.DataFrame(results).T

fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(results_df))
width = 0.35

ax.bar(x - width/2, results_df['accuracy'], width, label='Test Accuracy', alpha=0.8)
ax.bar(x + width/2, results_df['cv_mean'], width, label='CV Mean Accuracy', alpha=0.8)

ax.set_xlabel('Model')
ax.set_ylabel('Accuracy')
ax.set_title('Model Comparison - Breast Cancer Classification')
ax.set_xticks(x)
ax.set_xticklabels(results_df.index, rotation=45, ha='right')
ax.legend()
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print("\nModel Rankings by Test Accuracy:")
print(results_df.sort_values('accuracy', ascending=False))

## 8. Best Model - Detailed Evaluation

In [ ]:
# Get best model
best_model_name = max(results, key=lambda k: results[k]['accuracy'])
best_model = models[best_model_name]

print(f"Best Model: {best_model_name}")
print(f"Accuracy: {results[best_model_name]['accuracy']:.4f}\n")

# Detailed classification report
y_pred = best_model.predict(X_test_scaled)
print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=['Malignant', 'Benign']))

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Malignant', 'Benign'], yticklabels=['Malignant', 'Benign'])
plt.title(f'Confusion Matrix - {best_model_name}')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.show()

## 9. Further Optimization - Ensemble Methods

In [ ]:
# Hyperparameter tuning for Random Forest
param_grid_rf = {
    'n_estimators': [100, 200, 300],
    'max_depth': [10, 20, 30, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

rf_grid = GridSearchCV(RandomForestClassifier(random_state=42), param_grid_rf, cv=5, scoring='accuracy', n_jobs=-1)
rf_grid.fit(X_train_scaled, y_train)

print(f"Best Random Forest parameters: {rf_grid.best_params_}")
print(f"Best cross-validation score: {rf_grid.best_score_:.4f}")
print(f"Test set accuracy: {rf_grid.score(X_test_scaled, y_test):.4f}")

## 10. Summary and Conclusions

In [ ]:
print("=" * 60)
print("SUMMARY OF ACCURACY IMPROVEMENTS")
print("=" * 60)
print(f"\nBaseline KNN (default parameters): {baseline_score:.4f}")
print(f"Optimized KNN: {knn_grid.score(X_test_scaled, y_test):.4f}")
print(f"Best Overall Model ({best_model_name}): {results[best_model_name]['accuracy']:.4f}")
print(f"Optimized Random Forest: {rf_grid.score(X_test_scaled, y_test):.4f}")
print(f"\nImprovement over baseline: {(max(results[best_model_name]['accuracy'], rf_grid.score(X_test_scaled, y_test)) - baseline_score) * 100:.2f}%")
print("=" * 60)